# 👥 Notebook 03 — Customer Segmentation (RFM Analysis)

**RFM = Recency, Frequency, Monetary**

A proven marketing technique to segment customers based on their purchase behavior.

| Metric | Question |
|--------|----------|
| **Recency** | How recently did they buy? |
| **Frequency** | How often do they buy? |
| **Monetary** | How much do they spend? |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

DATA_PATH = '../data/'

orders      = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv',
                          parse_dates=['order_purchase_timestamp'])
order_items = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
customers   = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
payments    = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')

print('Data loaded ✅')

## 1. Build RFM Base Table

In [ ]:
# Filter to delivered orders only
delivered = orders[orders['order_status'] == 'delivered'].copy()

# Merge with customers and payments
df = delivered.merge(customers[['customer_id','customer_unique_id','customer_state']],
                     on='customer_id', how='left')

order_value = payments.groupby('order_id')['payment_value'].sum().reset_index()
df = df.merge(order_value, on='order_id', how='left')

# Reference date = 1 day after last purchase in dataset
ref_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f'Reference date: {ref_date.date()}')

# Compute RFM
rfm = df.groupby('customer_unique_id').agg(
    Recency   = ('order_purchase_timestamp', lambda x: (ref_date - x.max()).days),
    Frequency = ('order_id', 'nunique'),
    Monetary  = ('payment_value', 'sum')
).reset_index()

print(f'RFM table shape: {rfm.shape}')
rfm.describe()

## 2. Score Each Customer (1–5 scale)

In [ ]:
# Recency: lower = better → reverse score
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_Total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print(rfm[['customer_unique_id','Recency','Frequency','Monetary','R_score','F_score','M_score','RFM_Total']].head(10))

## 3. Assign Customer Segments

In [ ]:
def segment_customer(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Customers'
    elif r >= 3 and f <= 2 and m >= 3:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r == 1 and f >= 4:
        return 'Cant Lose Them'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
seg_counts = rfm['Segment'].value_counts()
print(seg_counts)

## 4. Visualize Segments

In [ ]:
colors = ['#2ecc71','#27ae60','#3498db','#2980b9','#e67e22','#e74c3c','#c0392b','#95a5a6']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart - customer count
axes[0].pie(seg_counts.values, labels=seg_counts.index,
            autopct='%1.1f%%', startangle=140, colors=colors[:len(seg_counts)])
axes[0].set_title('Customer Segment Distribution', fontsize=13, fontweight='bold')

# Bar chart - avg monetary per segment
seg_monetary = rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False)
bars = axes[1].bar(seg_monetary.index, seg_monetary.values,
                   color=colors[:len(seg_monetary)])
for bar, val in zip(bars, seg_monetary.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'R${val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('Avg Spend per Segment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Avg Monetary Value (R$)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../data/rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. RFM Heatmap (Recency vs Frequency)

In [ ]:
heatmap_data = rfm.groupby(['R_score','F_score'])['Monetary'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlGnBu',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Monetary (R$)'})
ax.set_title('RFM Heatmap — Avg Spend by Recency & Frequency', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency Score')
ax.set_ylabel('Recency Score')
plt.tight_layout()
plt.savefig('../data/rfm_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Export Segmented Customer List

In [ ]:
rfm.to_csv('../data/rfm_segments.csv', index=False)
print('Exported rfm_segments.csv ✅')
print(f'\nSegment Summary:')
print(rfm.groupby('Segment').agg(
    Count=('customer_unique_id','count'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean')
).round(1).sort_values('Avg_Monetary', ascending=False))

---
## ✅ RFM Segmentation Summary

| Segment | Strategy |
|---------|----------|
| **Champions** | Reward them. They're your best customers. |
| **Loyal Customers** | Upsell higher value products. |
| **At Risk** | Send win-back campaigns urgently. |
| **Lost** | Low priority — only attempt with deep discounts. |
| **Recent Customers** | Onboard well, build habits early. |

➡️ **Next:** Go to `04_Product_Analysis.ipynb`